# GNSS in Lunar Orbit
[1] S. Bhamidipati, T. Mina, and G. Gao, ‘Design Considerations of a Lunar Navigation Satellite System with Time-Transfer from Earth-GPS’, presented at the Proceedings of the 34th International Technical Meeting of the Satellite Division of The Institute of Navigation (ION GNSS+ 2021), Sep. 2021, pp. 950–965. doi: 10.33012/2021.18021.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

%load_ext autoreload
%autoreload 2
import pylupnt as pnt

flag_plot = True

## Setup Lunar Navigation Satellites

In [3]:
# Epoch(TAI)
t0_tai = pnt.convert_time(pnt.gregorian2time(2025, 11, 9, 0, 0, 0), pnt.UTC, pnt.TAI)
N_sc = 3

# Classical orbital elements (a, e, i, W, w, M) [km, -, rad, rad, rad, rad]
sma = [6541.4, 7500, 9750.5]  # [km] Semi-major axis
ecc = [0.6, 0.05, 0.7]  # [-] Eccentricity
inc = np.deg2rad([56.2, 40, 65.5])  # [rad] Inclination
asc = np.deg2rad([0, 0, 0])  # [rad] Right ascension of the ascending node
aop = np.deg2rad([90, 90, 90])  # [rad] Argument of periapsis
man = np.deg2rad([0, 0, 0])  # [rad] Mean anomaly

coe_op = np.zeros((N_sc, 6))
for i in range(N_sc):
    coe_op[i] = np.array([sma[i], ecc[i], inc[i], asc[i], aop[i], man[i]])

rv0_m2sc_op = pnt.classical2cart(coe_op, pnt.GM_MOON)
rv0_m2sc_ci = pnt.convert_frame(t0_tai, rv0_m2sc_op, pnt.MOON_OP, pnt.MOON_CI)

# Time
period = 2 * np.pi * np.sqrt(np.power(coe_op[0, 0], 3) / pnt.GM_MOON)  # [s] Orbital period
dt = 2 * pnt.SECS_MINUTE  # [s] Simulation time step
dt_prop = 15  # [s] Propagation time step
tf = 2 * 31 * pnt.SECS_DAY  # [s] Simulation final time
N_t = int(tf / dt)  # [-] Number of time steps
tspan = np.linspace(0, tf, N_t)  # [s] Time since first epoch
tspan_h = tspan / pnt.SECS_HOUR  # [h] Time since first epoch
tspan_day = tspan / pnt.SECS_DAY  # [day] Time since first epoch
t_tai = t0_tai + tspan  # [s] Epochs (TAI)

# Dynamics
dyn = pnt.NBodyDynamics()
dyn.add_body(pnt.Body.Moon(5, 5))
dyn.add_body(pnt.Body.Earth())
dyn.add_body(pnt.Body.Sun())
dyn.set_frame(pnt.MOON_CI)
dyn.set_time_step(dt_prop)

# Propagation
# rv_from2to_frame [km, km/s] (x, y, z, vx, vy, vz)
rv_m2sc_ci = np.zeros((N_sc+1, N_t, 6))
rv_m2sc_pa = np.zeros((N_sc+1, N_t, 6))
for i in range(N_sc):
    rv_m2sc_ci[i] = dyn.propagate(rv0_m2sc_ci[i], t0_tai, t_tai)
    rv_m2sc_pa[i] = pnt.convert_frame(t_tai, rv_m2sc_ci[i], pnt.MOON_CI, pnt.MOON_PA)

# Earth and Sun
rv_m2e_ci = pnt.get_body_pos_vel(t_tai, pnt.MOON, pnt.EARTH, pnt.MOON_CI)
rv_m2s_ci = pnt.get_body_pos_vel(t_tai, pnt.MOON, pnt.SUN, pnt.MOON_CI)
rv_m2e_pa = pnt.convert_frame(t_tai, rv_m2e_ci, pnt.MOON_CI, pnt.MOON_PA)
rv_m2s_pa = pnt.convert_frame(t_tai, rv_m2s_ci, pnt.MOON_CI, pnt.MOON_PA)
rv_e2s_eci = pnt.get_body_pos_vel(t_tai, pnt.EARTH, pnt.SUN, pnt.ECI)

## Lunar Transfer Trajectory
Optionally, we create a cislunar transfer trajectory from LEO to the lunar ELFO using pykep.
See the [pykep website](https://esa.github.io/pykep/installation.html) for the installation instructions.

In [ ]:
add_cislunar = True

if add_cislunar:
    import pykep as pk
    # Add a 4th satellite a straight line from the Earth to the Moon
    N_sc = 4
    
    coe_leo = [pnt.R_EARTH + 10e4, 0.001, np.deg2rad(0), np.deg2rad(270), 0.0, 0.0]  # start position (in MCI frame)
    rv0_transfer = pnt.classical2cart(coe_leo, pnt.GM_EARTH)
    rvf_transfer = rv_m2sc_ci[0, -1, :].flatten() - rv_m2e_ci[-1, :]   # (s - m) - (e - m) = s - e

    print("rv0: ", rv0_transfer)
    print("rvf: ", rvf_transfer)
                    
    # Transfer orbit
    l = pk.lambert_problem(rv0_transfer[:3] * 1000, rvf_transfer[:3] * 1000, tof=tspan[-1], mu=pk.MU_EARTH, max_revs=2)

    rev = 0
    v1 = np.array(l.get_v1()[rev])/1000
    v2 = np.array(l.get_v2()[rev])/1000

    print("v1: ", v1)

    rv0_transfer_eci = np.concatenate((rv0_transfer[:3], v1))
    rv0_transfer_mci = rv0_transfer_eci + rv_m2e_ci[0, :]   # (s - e) + (e - m) = s - m

    # propagate
    dyn_tb = pnt.NBodyDynamics()
    dyn_tb.add_body(pnt.Body.Earth())
    dyn_tb.set_frame(pnt.GCRF)
    dyn_tb.set_time_step(dt_prop)

    rv_transfer_eci = dyn_tb.propagate(rv0_transfer_eci, t0_tai, t_tai)
    rv_m2sc_ci[3] = rv_transfer_eci + rv_m2e_ci


## Plot Orbits

In [ ]:
if flag_plot:
    fig = go.Figure()
    pnt.plot.plot_orbits(fig, rv_m2sc_ci[:3, ::10])
    pnt.plot.plot_body(
        fig,
        pnt.MOON,
        size_factor=2,
        alpha=0.5,
    )
    pnt.plot.set_view(fig, -80, 20, 2.5)
    fig.update_layout(showlegend=True, width=400, height=400)
    fig.show()

In [ ]:
if flag_plot:
    fig = go.Figure()
    transfer_traj_mci = rv_m2sc_ci[3, ::10, :]
    transfer_traj_eci = rv_m2sc_ci[3, ::10, :] - rv_m2e_ci[::10, :]  # (s - m) - (e - m) = s - e   time x 6
    trajs = np.zeros((3, transfer_traj_eci.shape[0], transfer_traj_eci.shape[1]))
    trajs[0] = transfer_traj_eci     # transfer trajectory in ECI frame                  
    trajs[1] = rv_m2e_ci[::10, :]    # Moon trajectory in MCI frame
    trajs[2] = rv_m2sc_ci[0, ::10, :] - rv_m2e_ci[::10, :]   # Lunar satellite trajectory in ECI frame

    pnt.plot.plot_orbits(fig, trajs)
    pnt.plot.plot_body(
        fig,
        pnt.EARTH,
        size_factor=2,
        alpha=0.5,
    )
    pnt.plot.set_view(fig, -80, 20, 2.5)
    fig.update_layout(showlegend=True, width=400, height=400)
    fig.show()

## Load the TLE Files
We set the GNSS satellite orbits, by loading the TLE files and propagating it to the target epoch

In [25]:
dt_prop_gnss = 10 * 60  # [s] Propagation time step

# with J2
dyn_gnss = pnt.NBodyDynamics(pnt.IntegratorType.RKF45)
dyn_gnss.set_integrator_params(pnt.IntegratorParams(max_iter=20, abstol=1e-10, reltol=1e-10))
dyn_gnss.add_body(pnt.Body.Earth(2, 0))  # Earth
dyn_gnss.set_frame(pnt.ECI)
dyn_gnss.set_time_step(dt_prop_gnss)

### GPS

In [24]:
# gps
tles = pnt.TLE.from_file("gps_2025_01_01")
N_gps = len(tles)
coe_gps_eci = np.zeros((N_gps, N_t, 6))
rv_gps_eci = np.zeros((N_gps, N_t, 6))
rv_gps_ci = np.zeros((N_gps, N_t, 6))

prns = np.zeros(N_gps, dtype=int)
for i in range(N_gps):
    coe0_gps = pnt.tle2classical(tles[i], pnt.GM_EARTH)
    rv0_gps_eci = pnt.classical2cart(coe0_gps, pnt.GM_EARTH)    
    rv_gps_eci[i] = dyn_gnss.propagate(rv0_gps_eci, tles[i].epoch_tai, t_tai)
    rv_gps_ci[i] = pnt.convert_frame(t_tai, rv_gps_eci[i], pnt.ECI, pnt.MOON_CI)
    prns[i] = tles[i].prn

### Galileo

In [ ]:
# galileo
tles = pnt.TLE.from_file("galileo_2025_01_01")
N_gal = len(tles)
coe_gal_eci = np.zeros((N_gal, N_t, 6))
rv_gal_eci = np.zeros((N_gal, N_t, 6))
rv_gal_ci = np.zeros((N_gal, N_t, 6))

prns_gal = np.zeros(N_gal, dtype=int)
for i in range(N_gal):
    coe0_gal = pnt.tle2classical(tles[i], pnt.GM_EARTH)
    rv0_gal_eci = pnt.classical2cart(coe0_gal, pnt.GM_EARTH)
    rv_gal_eci[i] = dyn_gnss.propagate(rv0_gal_eci, tles[i].epoch_tai, t_tai)
    rv_gal_ci[i] = pnt.convert_frame(t_tai, rv_gal_eci[i], pnt.ECI, pnt.MOON_CI)
    prns_gal[i] = tles[i].prn

### QZSS

In [ ]:
tles = pnt.TLE.from_file("qzss_2025_01_01")
N_qzss = len(tles)
coe_qzss_eci = np.zeros((N_qzss, N_t, 6))
rv_qzss_eci = np.zeros((N_qzss, N_t, 6))
rv_qzss_ci = np.zeros((N_qzss, N_t, 6))
prns_qzss = np.zeros(N_qzss, dtype=int)
for i in range(N_qzss):
    coe0_qzss = pnt.tle2classical(tles[i], pnt.GM_EARTH)
    rv0_qzss_eci = pnt.classical2cart(coe0_qzss, pnt.GM_EARTH)
    rv_qzss_eci[i] = dyn_gnss.propagate(rv0_qzss_eci, tles[i].epoch_tai, t_tai)
    rv_qzss_ci[i] = pnt.convert_frame(t_tai, rv_qzss_eci[i], pnt.ECI, pnt.MOON_CI)
    prns_qzss[i] = tles[i].prn

print(prns_qzss)

### GNSS Orientation
Next we compute the orientation of the GPS satellites, where
- y: Facing perpendicular to the Sun
- z: Facing towards Earth

In [30]:
# gps
e_gps2e = pnt.normalize(-rv_gps_eci[:, :, :3])
e_gps2s = pnt.normalize(rv_e2s_eci[None, :, :3] - rv_gps_eci[:, :, :3])
ez_gps = e_gps2e
ey_gps = pnt.cross_norm(e_gps2e, e_gps2s)
ex_gps = pnt.cross_norm(ey_gps, ez_gps)

# galileo
e_gal2e = pnt.normalize(-rv_gal_eci[:, :, :3])
e_gal2s = pnt.normalize(rv_e2s_eci[None, :, :3] - rv_gal_eci[:, :, :3])
ez_gal = e_gal2e
ey_gal = pnt.cross_norm(e_gal2e, e_gal2s)
ex_gal = pnt.cross_norm(ey_gal, ez_gal)

# qzss
e_qzss2e = pnt.normalize(-rv_qzss_eci[:, :, :3])
e_qzss2s = pnt.normalize(rv_e2s_eci[None, :, :3] - rv_qzss_eci[:, :, :3])
ez_qzss = e_qzss2e
ey_qzss = pnt.cross_norm(e_qzss2e, e_qzss2s)
ex_qzss = pnt.cross_norm(ey_qzss, ez_qzss)


## Load Antenna Patterns
Next, we load the GNSS sidelobe antenna patterns and the lunar receiver antenna pattern

In [ ]:
# Read data from file
import os
import math

# gps
df = pd.read_csv(pnt.find_file('gps_table.csv'))
gps_antenna_names = df.set_index("PRN")["LM_File"].to_dict()
# for gps that LM file is not available, use the ACE_file instead
for prn in gps_antenna_names.keys():
    if type(gps_antenna_names[prn]) == float:
        gps_antenna_names[prn] = df.loc[prn-1, "ACE_File"]
print(gps_antenna_names)
gps_antennas = {k: pnt.Antenna(v) for k, v in gps_antenna_names.items()}

# galileo
print(prns_gal)
galileo_antennas = {k: pnt.Antenna("Galileo_E1") for k in prns_gal}

# qzss
qzss_names = ["1R", "02", "03", "04", "05", "06", "07"]
qzss_antennas = {k: pnt.Antenna("QZSS_" + qzss_names[k-1] + "_L1") for k in range(1, 5)}

rx_antenna = pnt.Antenna("moongpsr")

Plot the antenna patterns

In [ ]:
fig, axs = plt.subplots(3,2,figsize=(10,8))

# gps II-R
pnt.plot.plot_antenna_gain_patter_2D(antenna=gps_antennas[2], ax=axs[0][0])  # II-R
pnt.plot.plot_antenna_gain_patter_2D(antenna=gps_antennas[3], ax=axs[0][1])  # II-F
pnt.plot.plot_antenna_gain_patter_2D(antenna=gps_antennas[4], ax=axs[1][0])  # III
pnt.plot.plot_antenna_gain_patter_2D(antenna=galileo_antennas[5], ax=axs[1][1])  # Galileo
pnt.plot.plot_antenna_gain_patter_2D(antenna=qzss_antennas[2], ax=axs[2][0])  # QZSS 2
pnt.plot.plot_antenna_gain_patter_2D(antenna=rx_antenna, ax=axs[2][1]) # Moon GPS-R
plt.xlim(-5,5)
plt.ylim(10,15)


plt.tight_layout()
plt.legend()

## Plot GNSS Constellation

In [ ]:
t = 0
if flag_plot:
    tickvals = np.arange(-30, 31, 10) * 1e3
    fig = go.Figure()

    pnt.plot.plot_body(fig, pnt.EARTH, size_factor=5)

    # gps
    pnt.plot.plot_orbits(fig, rv_gps_eci[:, :int(pnt.SECS_DAY / dt):10], t=t, color="lightgray")
    for i in range(N_gps):
        pnt.plot.plot_frame(
            fig, rv_gps_eci[i, t, :3], np.vstack((ex_gps[i, t], ey_gps[i, t], ez_gps[i, t])), length=pnt.R_EARTH, width=5, tip=5
        )
    pnt.plot.plot_arrow3(fig, np.zeros(3), pnt.normalize(rv_e2s_eci[t]), length=3 * pnt.R_EARTH, width=5, color="orange", tip=10)

    # galileo
    pnt.plot.plot_orbits(fig, rv_gal_eci[:, :int(pnt.SECS_DAY / dt):10], t=t, color="brown")
    for i in range(N_gal):
        pnt.plot.plot_frame(
            fig, rv_gal_eci[i, t, :3], np.vstack((ex_gal[i, t], ey_gal[i, t], ez_gal[i, t])), length=pnt.R_EARTH, width=5, tip=5
        )
    pnt.plot.plot_arrow3(fig, np.zeros(3), pnt.normalize(rv_e2s_eci[t]), length=3 * pnt.R_EARTH, width=5, color="orange", tip=10)

    # fig.update_layout(width=400, height=400)

    # qzss
    pnt.plot.plot_orbits(fig, rv_qzss_eci[:, :int(pnt.SECS_DAY / dt):10], t=t, color="olive")
    for i in range(N_qzss):
        pnt.plot.plot_frame(
            fig, rv_qzss_eci[i, t, :3], np.vstack((ex_qzss[i, t], ey_qzss[i, t], ez_qzss[i, t])), length=pnt.R_EARTH, width=5, tip=5
        )
    pnt.plot.plot_arrow3(fig, np.zeros(3), pnt.normalize(rv_e2s_eci[t]), length=3 * pnt.R_EARTH, width=5, color="orange", tip=10)

    pnt.plot.set_view(fig, 50, 20, 2.5)
    fig.show()

## Visibility Computation
Next, we compute the number of GPS and GLONASS satellites that can be tracked from lunar satellites considering the blockage from Earth and Moon, and the C/N0 threshold of the receiver

In [36]:
def compute_visibility(
    r1: np.ndarray, r2: np.ndarray, R_body: float, r_body: np.ndarray = None
) -> np.ndarray:
    if r_body is None:
        r_body = np.zeros(3)
    r = r2 - r1
    r_norm = np.linalg.norm(r, axis=1)
    r1body = r1 - r_body
    r1body_norm = np.linalg.norm(r1body, axis=1)
    dot = np.einsum("ij,ij->i", -r1body, r)
    theta1 = np.arccos(np.clip(dot / r_norm / r1body_norm, -1, 1))
    theta2 = np.arcsin(np.clip(R_body / r1body_norm, -1, 1))
    visibility = np.ones(len(r1), dtype=bool)
    visibility[(theta1 < theta2) & (r_norm > r1body_norm)] = False
    return visibility

In [37]:
def compute_cn0(N_gnss, rv_gnss_ci, ex_gnss, ey_gnss, ez_gnss, gnss_antennas, prn_gnss, P_tx):
    # Visibility
    vis_sc2sc = np.ones((N_sc, N_sc, N_t), dtype=bool)
    vis_sc2gps = np.ones((N_sc, N_gnss, N_t), dtype=bool)
    dist_sc2sc = np.zeros((N_sc, N_sc, N_t))
    dist_sc2gps = np.zeros((N_sc, N_gnss, N_t))
    phi_gps2sc = np.zeros((N_sc, N_gnss, N_t))
    phi_sc2gps = np.zeros((N_sc, N_gnss, N_t))
    theta_gps2sc = np.zeros((N_sc, N_gnss, N_t))
    G_tx = np.zeros((N_sc, N_gnss, N_t))
    G_rx = np.zeros((N_sc, N_gnss, N_t))

    for i in range(N_sc):
        for j in range(i + 1, N_sc):
            vis_sc2sc[i, j] &= compute_visibility(
                rv_m2sc_ci[i, :, :3], rv_m2sc_ci[j, :, :3], pnt.R_MOON
            )
            vis_sc2sc[j, i] &= vis_sc2sc[i, j]
            dist_sc2sc[i, j] = np.linalg.norm(
                rv_m2sc_ci[i, :, :3] - rv_m2sc_ci[j, :, :3], axis=1
            )
            dist_sc2sc[j, i] = dist_sc2sc[i, j]

        for j in range(N_gnss):
            # moon blockage
            vis_sc2gps[i, j] &= compute_visibility(
                rv_m2sc_ci[i, :, :3], rv_gnss_ci[j, :, :3], pnt.R_MOON
            )

            # earth blockage
            vis_sc2gps[i, j] &= compute_visibility(
                rv_m2sc_ci[i, :, :3], rv_gnss_ci[j, :, :3], pnt.R_EARTH, rv_m2e_ci[:, :3]
            )

            dist_sc2gps[i, j] = np.linalg.norm(
                rv_m2sc_ci[i, :, :3] - rv_gnss_ci[j, :, :3], axis=1
            )

            u_gps2sc = pnt.normalize(rv_m2sc_ci[i, :, :3] - rv_gnss_ci[j, :, :3])
            phi_gps2sc[i, j] = np.arccos(
                np.clip(np.sum(u_gps2sc * ez_gnss[j], axis=-1), -1, 1)
            )
            theta_gps2sc[i, j] = np.arctan2(
                np.sum(u_gps2sc * ey_gnss[j], axis=-1), np.sum(u_gps2sc * ex_gnss[j], axis=-1)
            )

            u_sc2gps = -u_gps2sc
            u_sc2e = pnt.normalize(rv_m2e_ci[:, :3] - rv_m2sc_ci[i, :, :3])
            phi_sc2gps[i, j] = np.arccos(np.clip(np.sum(u_sc2e * u_sc2gps, axis=-1), -1, 1))

            G_tx[i, j] = gnss_antennas[prn_gnss[j]].compute_gain(theta_gps2sc[i, j], phi_gps2sc[i, j])
            G_rx[i, j] = rx_antenna.compute_gain(0, phi_sc2gps[i, j])

            G_tx[i,j][~vis_sc2gps[i, j]] = np. nan
            G_rx[i,j][~vis_sc2gps[i, j]] = np. nan

        vis_sc2sc[i, i, :] = False

    # Link budget
    freq = 1575.42e6  # [Hz] GPS L1 frequency
    L_ad = 0.0  # [dB] A/D converter loss
    L_atm = 0.0  # [dB] Atmospheric loss
    Nf = 2  # [dB] Noise figure
    Tsys = 113  # [K] System noise temperature
    # [dB] Free space loss
    L_fs = 20 * np.log10((4 * np.pi * dist_sc2gps) / (pnt.C / freq))
    # [dB] Transmitter antenna gain
    kb_db = 10 * np.log10(1.38e-23)  # [dB] Boltzmann constant
    
    CN0 = P_tx + G_tx + G_rx - L_atm - L_fs - L_ad - Nf - kb_db - 10 * np.log10(Tsys)

    return CN0

In [38]:
def compute_receiver_error(CN0):
    Bn = 0.5      # [Hz] Code loop Bandwidth
    D = 0.3       # [chip] Chip duration
    T = 20e-3    # [s] Coherent integration time
    Bfe = 26e6    # [Hz] Front-end bandwidth
    fc = 1.023e6  # [Hz] Spread code frequency 
    Tc = 1 / fc   # [s] Spread code period 
    Rc = 1.023e6  # [Hz] Chip rate

    CN0 = 10 ** (CN0 / 10)  # [-] Carrier-to-noise density ratio

    # Error contributions
    # Delay Lock Loop (DLL) error\ [m]

    if (D >= (np.pi * Rc / Bfe)):
      sigma = np.sqrt(Bn / (2.0 * CN0) * D * (1.0 + 2.0 / (T * CN0 * (2 - D))))
    elif D > (Rc / Bfe):
      tmp1 = Bn / (2.0 * CN0)
      tmp2 = 1.0 / (Bfe * Tc) + Bfe * Tc / (np.pi - 1) * pow((D - 1.0 / (Bfe * Tc)), 2)
      tmp3 = 1.0 + 2.0 / (T * CN0 * (2 - D))
      sigma = np.sqrt(tmp1 * tmp2 * tmp3)
    else:
      sigma = np.sqrt(Bn / (2.0 * CN0) * (1.0 / (Bfe * Tc)) * (1.0 + 1.0 / (T * CN0)))

    sigma = sigma * (pnt.C * Tc) * 1000;  # convert to meters

    return sigma

Actual computation happens below

In [39]:
# CN0 For each constellation
CN0_GPS = compute_cn0(N_gps, rv_gps_ci, ex_gps, ey_gps, ez_gps, gps_antennas, prns, P_tx=14.3)
CN0_GAL = compute_cn0(N_gal, rv_gal_ci, ex_gal, ey_gal, ez_gal, galileo_antennas, prns_gal, P_tx=14.0)
CN0_QZSS = compute_cn0(N_qzss, rv_qzss_ci, ex_qzss, ey_qzss, ez_qzss, qzss_antennas, prns_qzss, P_tx=14.1)

### Plot the Number of Tracked Satellites

In [ ]:
CN0_threshold = 15

figsize = (10,3)
plt.rcParams.update({"font.size": 10})
if flag_plot:
    # C/N0 for GPS PRN 7
    plt.figure(figsize=figsize)
    ylims = [5, 50]
    i = 0
    j = prns.tolist().index(7)
    y = CN0_GPS[i, j]
    plt.plot(tspan_day, y, "-o", linewidth=1, markersize=2, label=f"GPS PRN {prns[j]}") 
    plt.axhline(CN0_threshold, color="black", linestyle="--", label="Threshold")
    plt.xlim(tspan_day[0], tspan_day[-1])
    plt.ylim(ylims)
    plt.xlabel("Time [days]")
    plt.ylabel("C/N0 [dB-Hz]")
    plt.grid()
    plt.legend()
    plt.tight_layout()
    plt.show()

    # C/N0 for Galileo PRN 5
    plt.figure(figsize=figsize)
    ylims = [5, 50]
    i = 0
    j = prns_gal.tolist().index(5)
    y = CN0_GAL[i, j]
    plt.plot(tspan_day, y, "-o", linewidth=1, markersize=2, label=f"Galileo PRN {prns_gal[j]}", color="orange")
    plt.axhline(CN0_threshold, color="black", linestyle="--", label="Threshold")
    plt.xlim(tspan_day[0], tspan_day[-1])
    plt.ylim(ylims)
    plt.xlabel("Time [days]")
    plt.ylabel("C/N0 [dB-Hz]")
    plt.grid()
    plt.legend()
    plt.tight_layout()
    plt.show()

    # C/N0 for QZSS PRN 2
    plt.figure(figsize=figsize)
    ylims = [5, 50]
    i = 0
    j = prns_qzss.tolist().index(2)
    y = CN0_QZSS[i, j]
    plt.plot(tspan_day, y, "-o", linewidth=1, markersize=2, label=f"QZSS PRN {prns_qzss[j]}", color="green")
    plt.axhline(CN0_threshold, color="black", linestyle="--", label="Threshold")
    plt.xlim(tspan_day[0], tspan_day[-1])
    plt.ylim(ylims)
    plt.xlabel("Time [days]")
    plt.ylabel("C/N0 [dB-Hz]")
    plt.grid()
    plt.legend()
    plt.tight_layout()
    plt.show()
    

In [ ]:
if flag_plot:   
    # Total tracked satellites at moon sat 1
    plt.figure(figsize=figsize)
    for i in range(1):
        y = np.sum(CN0_GPS[i] > CN0_threshold, axis=0)
        plt.plot(tspan_day, y.T, "-o", label=f"Tracked GPS Satellites", markersize=2)
    plt.xlim(tspan_day[0], tspan_day[-1])
    plt.xlabel("Time [days]")
    plt.ylabel("Visible GPS satellites")
    plt.legend(loc="upper right")
    ylim = 14
    plt.yticks(np.arange(0, ylim + 1, 2))
    plt.ylim(0, ylim)
    plt.grid()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=figsize)
    for i in range(1):
        y = np.sum(CN0_GAL[i] > CN0_threshold, axis=0)
        plt.plot(tspan_day, y.T, "-o", label=f"Tracked Galileo Satellites", markersize=2, color="orange")
    plt.xlim(tspan_day[0], tspan_day[-1])
    plt.xlabel("Time [days]")
    plt.ylabel("Visible Galileo satellites")
    plt.legend(loc="upper right")
    ylim = 14
    plt.yticks(np.arange(0, ylim + 1, 2))
    plt.ylim(0, ylim)
    plt.grid()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=figsize)
    for i in range(1):
        y = np.sum(CN0_QZSS[i] > CN0_threshold, axis=0)
        plt.plot(tspan_day, y.T, "-o", label=f"Tracked QZSS Satellites", markersize=2, color="green")
    plt.xlim(tspan_day[0], tspan_day[-1])
    plt.xlabel("Time [days]")
    plt.ylabel("Visible QZSS satellites")
    plt.legend(loc="upper right")
    ylim = 14
    plt.yticks(np.arange(0, ylim + 1, 2))
    plt.ylim(0, ylim)
    plt.grid()
    plt.tight_layout()
    plt.show()

In [ ]:
if flag_plot:
# Tracked gps satellites at each moon satellite
    plt.figure(figsize=figsize)
    idx1 = int(5.8 * pnt.SECS_DAY // dt)
    idx2 = int(7.5 * pnt.SECS_DAY // dt)
    for i in range(4):
        y = np.sum(CN0_GPS[i] > CN0_threshold, axis=0)
        plt.plot(tspan_day[idx1:idx2], y.T[idx1:idx2], "-o", label=f"Moon Sat {i+1}", markersize=2)
    plt.xlim(tspan_day[idx1], tspan_day[idx2])
    plt.xlabel("Time [days]")
    plt.ylabel("Visible GPS satellites")
    plt.legend(loc="upper right")
    plt.yticks(np.arange(0, ylim + 1, 2))
    plt.ylim(0, ylim)
    plt.grid()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=figsize)
    idx1 = int(5.8 * pnt.SECS_DAY // dt)
    idx2 = int(7.5 * pnt.SECS_DAY // dt)
    for i in range(4):
        y = np.sum(CN0_GAL[i] > CN0_threshold, axis=0)
        plt.plot(tspan_day[idx1:idx2], y.T[idx1:idx2], "-o", label=f"Moon Sat {i+1}", markersize=2)
    plt.xlim(tspan_day[idx1], tspan_day[idx2])
    plt.xlabel("Time [days]")
    plt.ylabel("Visible Galileo satellites")
    plt.legend(loc="upper right")
    plt.yticks(np.arange(0, ylim + 1, 2))
    plt.ylim(0, ylim)
    plt.grid()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=figsize)
    idx1 = int(5.8 * pnt.SECS_DAY // dt)
    idx2 = int(7.5 * pnt.SECS_DAY // dt)
    for i in range(4):
        y = np.sum(CN0_QZSS[i] > CN0_threshold, axis=0)
        plt.plot(tspan_day[idx1:idx2], y.T[idx1:idx2], "-o", label=f"Moon Sat {i+1}", markersize=2)
    plt.xlim(tspan_day[idx1], tspan_day[idx2])
    plt.xlabel("Time [days]")
    plt.ylabel("Visible QZSS satellites")
    plt.legend(loc="upper right")
    plt.yticks(np.arange(0, ylim + 1, 2))
    plt.ylim(0, ylim)
    plt.grid()
    plt.tight_layout()
    plt.show()

## Compute GNSS Receiver Error 

In [45]:
sigma_gps = compute_receiver_error(CN0_GPS[CN0_GPS > CN0_threshold])
sigma_gal = compute_receiver_error(CN0_GAL[CN0_GAL > CN0_threshold])
sigma_qzss = compute_receiver_error(CN0_QZSS[CN0_QZSS > CN0_threshold])

In [ ]:
# Plot the histogram of sigma over the entire epoch
plt.figure(figsize=(10, 3))
bins = np.arange(0, 25, 0.1)
plt.hist(sigma_gps.flatten(), bins=bins, alpha=0.5, label="GPS")
plt.hist(sigma_gal.flatten(), bins=bins, alpha=0.5, label="Galileo")
plt.hist(sigma_qzss.flatten(), bins=bins, alpha=0.5, label="QZSS")
plt.xlabel("Range error from Receiver Noise [m]")
plt.ylabel("Frequency")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()